# Memory A/B — does replaying the previous view change the analyst?

Each of the seven analysts was run **twice**, over an identical window on an identical
release clock, with identical code, model (`claude-sonnet-5`), text channel (`cue` /
`statement`) and tool schema. The arms differ by exactly one thing:

| arm | what the analyst is given |
|---|---|
| **off** | evidence only — measurements + policy language |
| **on** | the same evidence, plus its own previous view (direction, conviction, falsifier) and an instruction to grade that call before forming a new one |

The analyst is not shown any new data in the `on` arm. Its previous view was formed at
`t-1` from `t-1` evidence, and the outcome that view was graded against is already in the
measurement block — the newest value of the level feature *is* the release the previous
call was scored on. So this opens no look-ahead surface; it only tells the analyst what it
previously said.

**Why the recorded runs in `reports/` are not the control.** They predate the
`missing_inputs` contract and its tool-schema field, so they differ from an `on` run by two
treatments rather than one. Both arms here carry that contract, so it cancels and memory is
isolated. The recorded runs appear below only as a loose sanity reference.

**Why the paired test is the primary statistic.** At ~120 releases per driver, the standard
error on a single IC is roughly `1/sqrt(n)` ≈ 0.09 — wide enough to swallow a real effect.
Both arms saw the *same dates* and the *same outcomes*, so differencing them date by date
cancels everything they share and leaves only the treatment. Reading two IC columns side by
side is the weaker comparison; the paired column is the test.

In [ ]:
import os, sys, json, math
import numpy as np
import pandas as pd

sys.path.insert(0, os.path.abspath(".."))
from src.layered.evaluation import ICEvaluator, load_run, required_ic
from src.layered.evaluation import report_quality as rq

AB_DIR = "../reports/ab"
RECORDED = {                      # loose reference only — a different code version
    "inflation": "../reports/sweep_sonnet.jsonl",
    "labor_tightness": "../reports/labor_tightness_sonnet.jsonl",
    "curve_slope": "../reports/curve_slope_sonnet.jsonl",
    "term_premium": "../reports/term_premium_sonnet.jsonl",
    "balance_sheet": "../reports/balance_sheet_sonnet.jsonl",
    "financial_conditions": "../reports/financial_conditions_sonnet.jsonl",
    "inflation_expectations": "../reports/inflation_expectations_sonnet.jsonl",
}

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)

# Pair up whatever has both arms on disk (discover_runs is non-recursive and will not
# see reports/ab/, which is deliberate — these must not pollute the recorded set).
DRIVERS = sorted({os.path.basename(p).rsplit("_", 1)[0]
                  for p in os.listdir(AB_DIR) if p.endswith(".jsonl")})
pairs = {}
for d in DRIVERS:
    off_p, on_p = f"{AB_DIR}/{d}_off.jsonl", f"{AB_DIR}/{d}_on.jsonl"
    if os.path.exists(off_p) and os.path.exists(on_p):
        pairs[d] = (load_run(off_p), load_run(on_p))

print(f"{len(pairs)} paired drivers loaded")
for d, (o, n) in pairs.items():
    print(f"  {d:<24} off {len(o.views):>4} releases   on {len(n.views):>4}   "
          f"{o.views.index.min().date()} -> {o.views.index.max().date()}")

## 1. Headline IC, both arms

`ic` is rank IC of signed conviction against the driver's move to the next release — the
same evaluator, clock and outcome the recorded runs used, so nothing here is a new metric.
`d_ic` is the raw difference (on − off); treat it as descriptive, not as a test.

In [ ]:
rows = []
for d, (off, on) in pairs.items():
    ev = ICEvaluator(off.level.dropna(), steps=1)          # identical level series in both arms
    r_off = ev.evaluate(off.signed, "off")
    r_on  = ev.evaluate(on.signed,  "on")
    rows.append({
        "driver": d,
        "n_off": r_off.n, "ic_off": round(r_off.ic, 3), "t_off": round(r_off.t_stat, 2),
        "hit_off": round(r_off.hit_rate, 3),
        "n_on": r_on.n, "ic_on": round(r_on.ic, 3), "t_on": round(r_on.t_stat, 2),
        "hit_on": round(r_on.hit_rate, 3),
        "d_ic": round(r_on.ic - r_off.ic, 3),
        "d_hit": round(r_on.hit_rate - r_off.hit_rate, 3),
    })
ic_table = pd.DataFrame(rows).set_index("driver").sort_values("d_ic", ascending=False)
ic_table

## 2. The paired test

For every release date both arms produced a signed conviction against the *same* realized
move `y`. The per-date contribution is `s * y` — positive when the call leaned the right
way, scaled by how strongly. The paired difference is

    delta_t = (s_on,t - s_off,t) * y_t

and a one-sample t-test on `delta` asks whether memory systematically improved alignment
with what actually happened. Everything the two arms share — the evidence, the outcome, the
regime, the driver's own predictability — cancels.

`agree` is the share of dates where the two arms gave the *same direction*. It is the
sanity check that has to be read first: if the arms agree almost always, memory barely
changed behaviour and no test on this sample can resolve anything, whatever the p-value says.

In [ ]:
rows = []
for d, (off, on) in pairs.items():
    ev = ICEvaluator(off.level.dropna(), steps=1)
    y = ev.outcome
    df = pd.concat([off.signed.rename("s_off"), on.signed.rename("s_on"), y.rename("y")],
                   axis=1).dropna()
    delta = (df["s_on"] - df["s_off"]) * df["y"]
    n = len(delta)
    sd = float(delta.std())
    t = float(delta.mean() / (sd / math.sqrt(n))) if n > 2 and sd > 0 else float("nan")
    rows.append({
        "driver": d, "n": n,
        "agree_dir": round(float((np.sign(df["s_on"]) == np.sign(df["s_off"])).mean()), 3),
        "mean_delta": round(float(delta.mean()), 5),
        "t_paired": round(t, 2),
        "conv_off": round(float(off.views["conviction"].mean()), 3),
        "conv_on": round(float(on.views["conviction"].mean()), 3),
    })
paired = pd.DataFrame(rows).set_index("driver").sort_values("t_paired", ascending=False)
paired

### Sign consistency across drivers

With ~120 dates each, one driver clearing |t| > 2 is weak evidence and one driver failing to
is not evidence of absence. Seven roughly independent analysts leaning the *same way* is a
stronger signal than any single t-statistic here. Under the null of no effect, the number of
positive `d_ic` out of seven is Binomial(7, 0.5) — 7/7 or 0/7 has a two-sided probability of
about 1.6%, 6/7 about 12.5%.

In [ ]:
pos_ic = int((ic_table["d_ic"] > 0).sum()); k = len(ic_table)
pos_t  = int((paired["t_paired"] > 0).sum())
binom_p = lambda k, n: round(2 * sum(math.comb(n, i) for i in range(k, n + 1)) / 2**n, 4)
print(f"drivers with d_ic  > 0 : {pos_ic}/{k}   two-sided binomial p ~ {binom_p(max(pos_ic, k-pos_ic), k)}")
print(f"drivers with t_paired>0: {pos_t}/{k}   two-sided binomial p ~ {binom_p(max(pos_t, k-pos_t), k)}")
print(f"\nmean d_ic across drivers      : {ic_table['d_ic'].mean():+.3f}")
print(f"median agreement between arms : {paired['agree_dir'].median():.1%}")

## 3. Calibration — is conviction doing work?

`ic_from_conviction` is the IC of signed conviction minus the IC of direction alone. Near
zero means the conviction ladder is not buying any ordering beyond the up/down call, and
sizing has headroom. This is where the memory arm is *supposed* to help: an analyst that has
just been shown it was wrong should size the next call differently.

In [ ]:
rows = []
for d, (off, on) in pairs.items():
    ev = ICEvaluator(off.level.dropna(), steps=1)
    c_off = ev.calibration_split(off.signed)
    c_on  = ev.calibration_split(on.signed)
    rows.append({
        "driver": d,
        "dir_only_off": c_off.loc["direction_only", "ic"],
        "signed_off":   c_off.loc["signed_conviction", "ic"],
        "conv_adds_off": c_off["ic_from_conviction"].iloc[0],
        "dir_only_on":  c_on.loc["direction_only", "ic"],
        "signed_on":    c_on.loc["signed_conviction", "ic"],
        "conv_adds_on": c_on["ic_from_conviction"].iloc[0],
    })
calib = pd.DataFrame(rows).set_index("driver")
calib["d_conv_adds"] = (calib["conv_adds_on"] - calib["conv_adds_off"]).round(3)
calib

---
# New outputs

Everything below grades output that did not exist before this change. These are not A/B
tests of memory — `missing_inputs` is present in **both** arms — but a read on whether the
new fields carry usable signal for the PM layer.

## 4. Declared gaps (`missing_inputs`)

Each analyst may name evidence it was **not** handed, as the driver that owns it. This is
deliberately kept out of the report prose and exempt from the cross-driver drift check: an
isolated analyst naming another driver here is the intended behaviour, not contamination.
The point is that the union of these across a meeting is a driver-to-driver dependency
graph — something no single isolated analyst can see and a PM must.

In [ ]:
rows = []
for d, (off, on) in pairs.items():
    for arm, run in (("off", off), ("on", on)):
        gaps = run.views["missing_inputs"]
        n = len(gaps)
        rows.append({"driver": d, "arm": arm,
                     "declares_any": round(float((gaps.str.len() > 0).mean()), 3),
                     "mean_named": round(float(gaps.str.len().mean()), 2),
                     "distinct_drivers": len({x for lst in gaps for x in lst})})
gaps_tbl = pd.DataFrame(rows).set_index(["driver", "arm"]).unstack()
gaps_tbl

In [ ]:
# The dependency graph: rows = the analyst writing, cols = the driver whose evidence it wants.
mat = pd.DataFrame(0, index=sorted(pairs), columns=sorted(pairs), dtype=int)
for d, (off, on) in pairs.items():
    for run in (off, on):
        for lst in run.views["missing_inputs"]:
            for tgt in lst:
                if tgt in mat.columns:
                    mat.loc[d, tgt] += 1
print("requests, both arms pooled (row = asking analyst, col = driver asked for)")
mat

In [ ]:
# What they actually said — the prose is the point, not the count. Capped per driver so
# one talkative analyst cannot crowd out the rest.
PER_DRIVER = 3
for d, (off, on) in pairs.items():
    seen = 0
    for run in (off, on):
        for line in open(run.path):
            if seen >= PER_DRIVER:
                break
            for m in (json.loads(line)["view"].get("missing_inputs") or []):
                print(f"[{d} -> {m['driver']}] {m['why']}")
                seen += 1
                break
    if seen == 0:
        print(f"[{d}] — declared no missing inputs in either arm")

## 5. Does conviction respond to being wrong?

`has_falsifier` only ever asked whether a falsifier was *written*. This asks the question no
single report can answer: having been wrong at the previous release, does the next call come
back softer or flipped — or at the same conviction, as though nothing happened?

Both halves come from the run file itself: the previous view's direction, and the realized
move, which is the change in `level` between consecutive records — the same quantity
`ICEvaluator` grades against. Degraded and carried rows are dropped first, matching how the
run is scored. `n_transitions` counts consecutive-release transitions *within* one arm; it is
not the number of paired comparisons between arms.

**Read this as a comparison between arms, not as an absolute score.** A memory-free analyst
also softens after a miss, because the move that made it wrong is visible in the new
evidence. The `off` column is exactly that baseline; the `on − off` difference is the part
attributable to being shown its own prior call.

In [ ]:
rows = []
for d, (off, on) in pairs.items():
    a, b = rq.conviction_response(off.path), rq.conviction_response(on.path)
    g = lambda x, k: x.get(k) if x.get(k) is not None else float("nan")
    rows.append({
        "driver": d,
        "n_trans_off": g(a, "n_transitions"), "n_trans_on": g(b, "n_transitions"),
        "wrong_off": g(a, "d_conv_after_wrong"), "wrong_on": g(b, "d_conv_after_wrong"),
        "right_off": g(a, "d_conv_after_right"), "right_on": g(b, "d_conv_after_right"),
        "gap_off": round(g(a, "d_conv_after_right") - g(a, "d_conv_after_wrong"), 3),
        "gap_on":  round(g(b, "d_conv_after_right") - g(b, "d_conv_after_wrong"), 3),
        "flip_wrong_off": g(a, "flip_rate_after_wrong"), "flip_wrong_on": g(b, "flip_rate_after_wrong"),
    })
resp = pd.DataFrame(rows).set_index("driver")
resp["d_gap"] = (resp["gap_on"] - resp["gap_off"]).round(3)
resp

## 6. Report quality, both arms

The pre-existing lexical checks, plus the two new ones. `cross_driver` is coarse and noisy by
construction (substring matching, no word boundaries — `"eas"` fires on "increase"), so read
it as a flag for review rather than proof of contamination. What matters here is that it did
not get *worse* in the `on` arm: the memory block must not push the analyst off its own driver.

In [ ]:
rows = []
for d, (off, on) in pairs.items():
    for arm, run in (("off", off), ("on", on)):
        q = rq.evaluate_run(run.path, driver=d)
        q["driver"], q["arm"] = d, arm
        rows.append(q)
quality = (pd.DataFrame(rows)
           .set_index(["driver", "arm"])
           [["n", "dir_consistent", "names_trade", "hallucinated", "cites_text",
             "cross_driver", "has_falsifier", "declares_gaps",
             "overconfident_given_gaps", "med_words"]])
quality

## 7. Sanity reference: the recorded runs

Confounded — the recorded runs lack the `missing_inputs` contract and its schema field, so a
difference here mixes two treatments. Included only to check that the `off` arm did not move
wildly from what was recorded, which would suggest something other than the treatment changed.

In [ ]:
rows = []
for d, (off, on) in pairs.items():
    p = RECORDED.get(d)
    if not p or not os.path.exists(p):
        continue
    rec = load_run(p)
    ev_r = ICEvaluator(rec.level.dropna(), steps=1)
    ev_o = ICEvaluator(off.level.dropna(), steps=1)
    rows.append({"driver": d,
                 "ic_recorded": round(ev_r.evaluate(rec.signed).ic, 3),
                 "n_recorded": len(rec.views),
                 "ic_off": round(ev_o.evaluate(off.signed).ic, 3),
                 "n_off": len(off.views)})
pd.DataFrame(rows).set_index("driver")

## 8. Verdict

In [ ]:
mean_d   = ic_table["d_ic"].mean()
agree    = paired["agree_dir"].median()
# Index off `paired`, not `ic_table` — the two frames are sorted differently, and masking
# one frame's index with the other's boolean would silently mislabel drivers.
sig      = paired.index[paired["t_paired"].abs() > 2].tolist()
gap_up   = int((resp["d_gap"] > 0).sum())

print(f"Drivers                     : {len(pairs)}")
print(f"Mean IC change (on - off)   : {mean_d:+.3f}")
print(f"Drivers with d_ic > 0       : {pos_ic}/{k}  (binomial p ~ {binom_p(max(pos_ic, k-pos_ic), k)})")
print(f"Median direction agreement  : {agree:.1%}")
print(f"Paired |t| > 2              : {sig if sig else 'none'}")
print(f"Calibration gap widened in  : {gap_up}/{len(resp)} drivers")
print()
if agree > 0.95:
    print("READ FIRST: the arms agree on direction almost always, so memory barely changed")
    print("behaviour. Any IC difference on this sample is noise, and the honest conclusion is")
    print("'no detectable effect', not 'no effect'.")
elif not sig:
    print("No individual driver clears |t| > 2 on the paired test. With ~120 dates per driver")
    print("only a large effect could resolve, so this is 'not detected', not 'not there'.")
    print(f"The informative read is the sign count: {pos_ic}/{k} drivers improved, mean d_ic")
    print(f"{mean_d:+.3f}. Judge that against the binomial p above, not against any one driver.")
else:
    print(f"Paired effect detected on: {', '.join(sig)}.")
    print("Check that d_ic agrees in sign with t_paired for those drivers, and whether the")
    print("rest of the panel leans the same way — one driver in seven at |t|>2 is what you")
    print("would expect by chance at the 5% level.")

### Caveats

1. **The `on` arm adds two things at once** — the replayed view *and* an instruction to
   self-assess. A difference could come from either. Separating them needs a placebo arm
   (same instruction, no actual prior view), which was not run.
2. **~120 releases per driver.** The paired design is far more powerful than comparing two
   ICs, but a small effect will not resolve on any single analyst. Sign consistency across
   the seven is the more informative read.
3. **`balance_sheet` IC is inflated by trend**, not skill — WALCL moves near-deterministically
   through QE/QT, so "higher next month?" is close to free. Do not use it as the headline case.
4. **`cross_driver` is lexical and noisy**; a level shift between arms means little, a large
   jump is worth reading reports for.
5. **`missing_inputs` was used sparsely** in early testing (2 of 12 records). If the
   dependency graph above is thin, that is a prompt-strength question, not evidence that the
   analysts have no dependencies.